Sempre que for utilizar a simulação, fazer o upload dos arquivos do certificado do IoT Core

In [ ]:
!pip install paho-mqtt

In [ ]:
import paho.mqtt.client as mqtt
import json
import time
import random
import ssl
from datetime import datetime

In [ ]:
# ==============================================================================
# CONFIGURAÇÕES AWS IOT CORE
# ==============================================================================
# Substitua pelo seu endpoint do AWS IoT Core (encontrado em Settings no console)
AWS_ENDPOINT = "arl4q7tru8xla-ats.iot.us-east-1.amazonaws.com"
PORT = 8883 # Porta padrão para MQTT com TLS na AWS
SUBESTACAO_ID = "SUB-01"

# Caminho dos certificados (faça upload no Colab e ajuste os nomes)
ROOT_CA_PATH = "AmazonRootCA1.pem"
CERT_PATH = "device-certificate.pem.crt"
KEY_PATH = "device-private.pem.key"

# Tópicos
TOPIC_ACESSO = f"subestacao/{SUBESTACAO_ID}/acesso/request"
TOPIC_PORTA = f"subestacao/{SUBESTACAO_ID}/sensor/porta"
TOPIC_PRESENCA = f"subestacao/{SUBESTACAO_ID}/sensor/presenca"
TOPIC_AMBIENTE = f"subestacao/{SUBESTACAO_ID}/sensor/ambiente" # Novo tópico

In [ ]:
# ==============================================================================
# FUNÇÕES DE SIMULAÇÃO
# ==============================================================================
def simular_leitura_rfid():
    uids = ["A1B2C3D4", "F5E6D7C8", "11223344", "UID_FALSO"]
    return {
        "rfid_uid": random.choice(uids),
        "leitor_id": "RFID-PORTAO-01",
        "nivel_bateria_leitor_percent": round(random.uniform(80.0, 100.0), 1),
        "latencia_leitura_ms": random.randint(15, 45),
        "timestamp": datetime.utcnow().isoformat() + "Z"
    }

def simular_sensor_porta():
    return {
        "porta_id": "PORTA-PRINCIPAL",
        "status": random.choice(["ABERTA", "FECHADA"]),
        "tempo_permanencia_aberta_s": random.randint(0, 120),
        "tensao_trava_v": round(random.uniform(23.5, 24.5), 2), # Trava de 24V
        "timestamp": datetime.utcnow().isoformat() + "Z"
    }

def simular_sensor_presenca():
    return {
        "zona_id": "ZONA-ALTA-TENSAO",
        "detectado": True,
        "nivel_confianca_percent": round(random.uniform(90.0, 99.9), 1),
        "timestamp": datetime.utcnow().isoformat() + "Z"
    }

def simular_sensor_ambiente():
    # Novo payload ideal para séries temporais no InfluxDB
    return {
        "temperatura_c": round(random.uniform(25.0, 45.0), 2),
        "umidade_percent": round(random.uniform(30.0, 70.0), 1),
        "fator_potencia": round(random.uniform(0.92, 0.99), 2),
        "qualidade_sinal_wifi_dbm": random.randint(-85, -40),
        "timestamp": datetime.utcnow().isoformat() + "Z"
    }

In [ ]:
# ==============================================================================
# CONFIGURAÇÃO MQTT
# ==============================================================================
def on_connect(client, userdata, flags, rc):
    if rc == 0:
        print("✅ Conectado ao AWS IoT Core com sucesso!")
    else:
        print(f"❌ Falha na conexão. Código: {rc}")

client = mqtt.Client(client_id="Gateway-ESP32-Simulador")
client.on_connect = on_connect

# Configuração do TLS para AWS
client.tls_set(ca_certs=ROOT_CA_PATH, certfile=CERT_PATH, keyfile=KEY_PATH, cert_reqs=ssl.CERT_REQUIRED, tls_version=ssl.PROTOCOL_TLSv1_2, ciphers=None)

print("Tentando conectar à AWS...")
client.connect(AWS_ENDPOINT, PORT, 60)
client.loop_start()

try:
    for i in range(15):
        time.sleep(2)
        evento = random.choice(["rfid", "porta", "presenca", "ambiente"])

        if evento == "rfid":
            dados = simular_leitura_rfid()
            client.publish(TOPIC_ACESSO, json.dumps(dados), qos=1)
            print(f"🔑 [RFID] Payload: {dados}")

        elif evento == "porta":
            dados = simular_sensor_porta()
            client.publish(TOPIC_PORTA, json.dumps(dados), qos=1)
            print(f"🚪 [PORTA] Payload: {dados}")

        elif evento == "presenca":
            dados = simular_sensor_presenca()
            client.publish(TOPIC_PRESENCA, json.dumps(dados), qos=1)
            print(f"🚶 [PRESENÇA] Payload: {dados}")

        elif evento == "ambiente":
            dados = simular_sensor_ambiente()
            client.publish(TOPIC_AMBIENTE, json.dumps(dados), qos=1)
            print(f"🌡️ [AMBIENTE] Payload: {dados}")

except KeyboardInterrupt:
    print("\n🛑 Simulação interrompida.")
finally:
    client.loop_stop()
    client.disconnect()
    print("🔌 Desconectado.")

/tmp/ipykernel_22723/4138988630.py:10: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client(client_id="Gateway-ESP32-Simulador")


Tentando conectar à AWS...
✅ Conectado ao AWS IoT Core com sucesso!


/tmp/ipykernel_22723/3331824388.py:20: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat() + "Z"


🚪 [PORTA] Payload: {'porta_id': 'PORTA-PRINCIPAL', 'status': 'FECHADA', 'tempo_permanencia_aberta_s': 0, 'tensao_trava_v': 24.02, 'timestamp': '2026-06-02T00:08:28.104153Z'}


/tmp/ipykernel_22723/3331824388.py:38: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat() + "Z"


🌡️ [AMBIENTE] Payload: {'temperatura_c': 26.14, 'umidade_percent': 47.7, 'fator_potencia': 0.92, 'qualidade_sinal_wifi_dbm': -47, 'timestamp': '2026-06-02T00:08:30.104979Z'}
🚪 [PORTA] Payload: {'porta_id': 'PORTA-PRINCIPAL', 'status': 'FECHADA', 'tempo_permanencia_aberta_s': 118, 'tensao_trava_v': 23.8, 'timestamp': '2026-06-02T00:08:32.105600Z'}
🌡️ [AMBIENTE] Payload: {'temperatura_c': 37.87, 'umidade_percent': 30.3, 'fator_potencia': 0.93, 'qualidade_sinal_wifi_dbm': -62, 'timestamp': '2026-06-02T00:08:34.106260Z'}


/tmp/ipykernel_22723/3331824388.py:28: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat() + "Z"


🚶 [PRESENÇA] Payload: {'zona_id': 'ZONA-ALTA-TENSAO', 'detectado': True, 'nivel_confianca_percent': 96.6, 'timestamp': '2026-06-02T00:08:36.106809Z'}
🚶 [PRESENÇA] Payload: {'zona_id': 'ZONA-ALTA-TENSAO', 'detectado': True, 'nivel_confianca_percent': 94.8, 'timestamp': '2026-06-02T00:08:38.107419Z'}
🚶 [PRESENÇA] Payload: {'zona_id': 'ZONA-ALTA-TENSAO', 'detectado': True, 'nivel_confianca_percent': 93.8, 'timestamp': '2026-06-02T00:08:40.108193Z'}
🚪 [PORTA] Payload: {'porta_id': 'PORTA-PRINCIPAL', 'status': 'ABERTA', 'tempo_permanencia_aberta_s': 113, 'tensao_trava_v': 23.88, 'timestamp': '2026-06-02T00:08:42.108644Z'}
🚪 [PORTA] Payload: {'porta_id': 'PORTA-PRINCIPAL', 'status': 'FECHADA', 'tempo_permanencia_aberta_s': 26, 'tensao_trava_v': 23.65, 'timestamp': '2026-06-02T00:08:44.109389Z'}
🚶 [PRESENÇA] Payload: {'zona_id': 'ZONA-ALTA-TENSAO', 'detectado': True, 'nivel_confianca_percent': 98.2, 'timestamp': '2026-06-02T00:08:46.109801Z'}
🌡️ [AMBIENTE] Payload: {'temperatura_c': 30.6, 'um

/tmp/ipykernel_22723/3331824388.py:11: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat() + "Z"


🔑 [RFID] Payload: {'rfid_uid': '11223344', 'leitor_id': 'RFID-PORTAO-01', 'nivel_bateria_leitor_percent': 91.9, 'latencia_leitura_ms': 17, 'timestamp': '2026-06-02T00:08:50.111039Z'}
🚪 [PORTA] Payload: {'porta_id': 'PORTA-PRINCIPAL', 'status': 'FECHADA', 'tempo_permanencia_aberta_s': 27, 'tensao_trava_v': 24.48, 'timestamp': '2026-06-02T00:08:52.111630Z'}
🚶 [PRESENÇA] Payload: {'zona_id': 'ZONA-ALTA-TENSAO', 'detectado': True, 'nivel_confianca_percent': 98.8, 'timestamp': '2026-06-02T00:08:54.112049Z'}
🚶 [PRESENÇA] Payload: {'zona_id': 'ZONA-ALTA-TENSAO', 'detectado': True, 'nivel_confianca_percent': 97.5, 'timestamp': '2026-06-02T00:08:56.112731Z'}
🔌 Desconectado.
